# 01 Data Loading

In this notebook, we load the **Reuter_50_50** dataset from the local folder structure and convert it into a structured pandas DataFrame.

The main goals of this notebook are:

- to read all text documents from the `C50train` and `C50test` folders,
- to extract the author name and dataset split (`train` or `test`),
- to store the raw text together with metadata,
- and to perform an initial structural check of the dataset.

The output of this notebook will be a clean tabular representation of the dataset that will be used in the following notebooks for exploratory analysis, preprocessing, feature extraction, and modeling.

In [1]:
from pathlib import Path
import pandas as pd

In [3]:
DATA_DIR = Path("../data")
TRAIN_DIR = DATA_DIR / "C50train"
TEST_DIR = DATA_DIR / "C50test"

print("DATA_DIR:", DATA_DIR.resolve())
print("TRAIN_DIR exists:", TRAIN_DIR.exists())
print("TEST_DIR exists:", TEST_DIR.exists())

DATA_DIR: C:\Users\Dell\Desktop\zaga-ip2\data
TRAIN_DIR exists: True
TEST_DIR exists: True


In [4]:
def load_split(split_dir, split_name):
    records = []

    for author_dir in sorted(split_dir.iterdir()):
        if not author_dir.is_dir():
            continue

        author = author_dir.name

        for file_path in sorted(author_dir.glob("*.txt")):
            try:
                text = file_path.read_text(encoding="utf-8", errors="ignore")
            except Exception as e:
                print(f"Error reading {file_path}: {e}")
                continue

            records.append({
                "file_name": file_path.name,
                "author": author,
                "split": split_name,
                "text": text
            })

    return records

In [6]:
train_records = load_split(TRAIN_DIR, "train")
test_records = load_split(TEST_DIR, "test")

print("Train documents:", len(train_records))
print("Test documents:", len(test_records))
print("Total documents:", len(train_records) + len(test_records))

Train documents: 2500
Test documents: 2500
Total documents: 5000


## Initial dataset loading

At this stage, we defined the dataset paths and created a helper function for loading documents from the directory structure.

The dataset is organized into two main parts:

- `C50train` — training documents
- `C50test` — test documents

Inside each of these folders, there are subfolders named after authors, and each subfolder contains text files written by that author. This structure allows us to directly derive:

- the **author label** from the folder name,
- the **dataset split** from the parent folder,
- and the **raw text** from the file content.

The initial counts of loaded documents are important because they confirm whether the dataset has been read correctly. Based on the dataset description, we expect:

- **2500 training documents**
- **2500 test documents**
- **5000 documents in total**

If these values match, we can proceed confidently to the construction and inspection of the final DataFrame.

In [9]:
df = pd.DataFrame(train_records + test_records)

print(df.shape)
df.head()

(5000, 4)


,file_name,author,split,text
0,106247newsML.txt,AaronPressman,train,The Internet may be overflowing with new techn...
1,120600newsML.txt,AaronPressman,train,The U.S. Postal Service announced Wednesday a ...
2,120683newsML.txt,AaronPressman,train,Elementary school students with access to the ...
3,136958newsML.txt,AaronPressman,train,An influential Internet organisation has backe...
4,137498newsML.txt,AaronPressman,train,An influential Internet organisation has backe...


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype 
---  ------     --------------  ----- 
 0   file_name  5000 non-null   object
 1   author     5000 non-null   object
 2   split      5000 non-null   object
 3   text       5000 non-null   object
dtypes: object(4)
memory usage: 156.4+ KB


In [11]:
df.isna().sum()

file_name    0
author       0
split        0
text         0
dtype: int64

## DataFrame construction and structural inspection

After loading all text files, the next step is to combine the extracted records into a single pandas DataFrame.

The resulting table should contain one row per document and the following fields:

- `file_name` — the original file name,
- `author` — the author label derived from the folder name,
- `split` — whether the document belongs to the training or test subset,
- `text` — the raw textual content of the document.

Inspecting the structure of the DataFrame is important for several reasons:

1. It confirms that the data has been loaded into the expected tabular format.
2. It helps verify the number of rows and columns.
3. It shows whether the data types are appropriate.
4. It allows us to check for missing values before continuing with later stages of analysis.

At this point, we expect the dataset to contain **5000 rows** and **4 columns**, with no missing values in the key fields. If this is confirmed, the dataset is ready for deeper inspection and quality checks.

## DataFrame structure verification

After combining the records from the training and test folders, the dataset was converted into a pandas DataFrame.

The resulting dataset contains:

- **5000 documents**
- **4 columns**

The columns represent the following information:

- **file_name** – the original name of the text document,
- **author** – the author label extracted from the folder name,
- **split** – indicates whether the document belongs to the training or test subset,
- **text** – the raw textual content of the document.

A structural inspection using `df.info()` confirms that all columns contain **5000 non-null values**, meaning that the dataset does **not contain missing values**.

This is important because missing values could complicate later stages of preprocessing and modeling. Since no missing values are present, the dataset is already clean from a structural perspective.

At this stage, the dataset is correctly loaded and ready for further exploration and validation of its distribution across authors and dataset splits.

In [12]:
df["author"].nunique()

50

In [13]:
df["split"].value_counts()

split
train    2500
test     2500
Name: count, dtype: int64

In [14]:
df["author"].value_counts().head(10)

author
AaronPressman       100
AlanCrosby          100
AlexanderSmith      100
BenjaminKangLim     100
BernardHickey       100
BradDorfman         100
DarrenSchuettler    100
DavidLawder         100
EdnaFernandes       100
EricAuchard         100
Name: count, dtype: int64

## Dataset distribution analysis

Next, we analyze the distribution of documents across authors and dataset splits.

First, we verify the number of unique authors in the dataset. The result shows that the dataset contains **50 different authors**, which means that the classification task will involve **50 classes**.

We then inspect the distribution between the training and test subsets. The dataset is evenly divided:

- **2500 training documents**
- **2500 test documents**

Finally, we examine the number of documents written by each author. The results show that each author contributes **exactly 100 documents**.

This means the dataset is **perfectly balanced**, which is highly beneficial for machine learning models because it prevents bias toward particular classes. Balanced datasets allow classification algorithms to learn patterns for each class more fairly and typically lead to more reliable evaluation results.

Overall, the dataset has the following characteristics:

- **Total documents:** 5000  
- **Number of authors (classes):** 50  
- **Documents per author:** 100  
- **Training documents:** 2500  
- **Test documents:** 2500  

These properties make the dataset well suited for authorship classification experiments.

In [15]:
df["text_length"] = df["text"].str.len()

df["text_length"].describe()

count    5000.000000
mean     3102.963800
std       826.238099
min       348.000000
25%      2610.750000
50%      3124.500000
75%      3549.500000
max      8811.000000
Name: text_length, dtype: float64

In [16]:
df.groupby("author")["text_length"].mean().sort_values(ascending=False).head(10)

author
PeterHumphrey       3630.14
LynnleyBrowning     3588.84
TheresePoletti      3578.01
SarahDavison        3564.01
KirstinRidley       3562.29
JaneMacartney       3528.19
AlexanderSmith      3497.61
LynneO'Donnell      3463.53
SamuelPerry         3454.93
DarrenSchuettler    3416.11
Name: text_length, dtype: float64

## Document length analysis

To better understand the structure of the dataset, we analyze the length of the documents.

The document length was calculated as the number of characters in each text. The descriptive statistics show the following:

- **Average length:** ~3103 characters  
- **Minimum length:** 348 characters  
- **Maximum length:** 8811 characters  
- **Standard deviation:** ~826 characters  

Most documents fall roughly between **2600 and 3500 characters**, indicating that the dataset contains moderately sized news articles with relatively consistent lengths.

We also computed the **average document length per author**. Some authors tend to write slightly longer texts than others. For example, authors such as *Peter Humphrey*, *Lynnley Browning*, and *Therese Poletti* have the highest average document lengths.

These differences may reflect stylistic characteristics of the authors, which can potentially help machine learning models distinguish between them in authorship classification tasks.

In [17]:
df["word_count"] = df["text"].str.split().str.len()

df["word_count"].describe()

count    5000.000000
mean      505.660400
std       135.473621
min        51.000000
25%       425.000000
50%       510.000000
75%       579.000000
max      1499.000000
Name: word_count, dtype: float64

In [18]:
df["word_count"].mean()

np.float64(505.6604)

## Word count analysis

In addition to character length, we also examine the number of words in each document.

The analysis shows that:

- **Average number of words:** ~506
- **Minimum:** 51
- **Maximum:** 1499
- **Standard deviation:** ~135

Most documents contain between **425 and 579 words**, indicating that the dataset consists of medium-length news articles with relatively consistent size.

This consistency is beneficial for text mining and machine learning tasks because extremely short or extremely long documents can introduce bias into feature extraction methods such as TF-IDF or Bag-of-Words.

Since the documents in this dataset have comparable lengths, the models will primarily learn **stylistic differences between authors** rather than being influenced by document size.

In [19]:
df.duplicated().sum()

np.int64(0)

In [20]:
from pathlib import Path

OUTPUT_DIR = Path("../processed")
OUTPUT_DIR.mkdir(exist_ok=True)

df.to_csv(OUTPUT_DIR / "reuters_raw_documents.csv", index=False)

print("Dataset saved to:", OUTPUT_DIR / "reuters_raw_documents.csv")
print("Shape:", df.shape)

Dataset saved to: ..\processed\reuters_raw_documents.csv
Shape: (5000, 6)


In [21]:
pd.read_csv("../processed/reuters_raw_documents.csv").head()

,file_name,author,split,text,text_length,word_count
0,106247newsML.txt,AaronPressman,train,The Internet may be overflowing with new techn...,2009,319
1,120600newsML.txt,AaronPressman,train,The U.S. Postal Service announced Wednesday a ...,2604,416
2,120683newsML.txt,AaronPressman,train,Elementary school students with access to the ...,491,72
3,136958newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2907,463
4,137498newsML.txt,AaronPressman,train,An influential Internet organisation has backe...,2299,356


## Final dataset summary

At the end of the data loading stage, the dataset has been successfully converted from the original folder-based structure into a structured tabular format.

The final dataset contains:

- **5000 documents**
- **50 authors**
- **2500 training documents**
- **2500 test documents**

Each document includes the following information:

- file name
- author label
- dataset split (train/test)
- raw text content
- text length (characters)
- word count

The dataset was checked for:

- missing values
- duplicate records
- structural consistency

No missing values or duplicates were detected.

Finally, the dataset was saved as a CSV file (`reuters_raw_documents.csv`) so it can be reused in the following notebooks without needing to reload the raw text files.

This concludes the **data loading and initial dataset inspection stage**. In the next notebook, we will perform **exploratory data analysis (EDA)** and visualize important characteristics of the dataset before applying text preprocessing and machine learning models.